<a href="https://colab.research.google.com/github/Ayushjha02/Data_Glacier/blob/Vc_copy_new/MovieRecommandationsystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DATA GATHERING AND PREPROCESSING**

*   Imported all required Library
*   Preprocessed all Data

In [1]:
import pandas as pd
from google.colab import drive
import numpy as np

In [2]:
## Mount google drive
drive.mount('/content/drive/')


Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
## Reading Data
movie = pd.read_csv('/content/drive/MyDrive/bollywood_data_set.csv')

In [4]:
movie = movie.drop(columns=['Unnamed: 0'])
movie['year_of_release'] = movie['year_of_release'].fillna("Unknown")
movie['actors'] = movie['actors'].fillna("Unknown")

In [5]:
# Clean 'IMDB_rating'
movie['IMDB_rating'] = movie['IMDB_rating'].replace(' ', np.nan)
movie['IMDB_rating'] = pd.to_numeric(movie['IMDB_rating'], errors='coerce')
movie['IMDB_rating'] = movie['IMDB_rating'].fillna(movie['IMDB_rating'].mean())

# Clean 'no_of_votes'
movie['no_of_votes'] = movie['no_of_votes'].replace(' ', np.nan)
movie['no_of_votes'] = pd.to_numeric(movie['no_of_votes'].str.replace(",", ""), errors='coerce')
movie['no_of_votes'] = movie['no_of_votes'].fillna(0).astype(int)

In [6]:
movie['director'] = movie['director'].apply(lambda x: x.split(", ") if x != "Unknown" else [])


In [7]:
!pip install yake
import yake

kw_extractor = yake.KeywordExtractor(top=5, stopwords=None)

def extract_keywords(text):
    try:
        keywords = kw_extractor.extract_keywords(text)
        return [kw for kw, score in keywords]
    except:
        return []

movie['keywords'] = movie['plot_description'].apply(extract_keywords)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 2.0 MB/s eta 0:00:00


**Creating Seprate CSV for each of the required node**

In [8]:
# Movies
movies = movie[['imdb-id', 'movie_name', 'year_of_release', 'runtime', 'IMDB_rating', 'no_of_votes', 'plot_description']].drop_duplicates()
movies.rename(columns={
    'imdb-id': 'imdb_id',
    'movie_name': 'title',
    'year_of_release': 'year',
    'IMDB_rating': 'rating',
    'no_of_votes': 'votes',
    'plot_description': 'plot'
}, inplace=True)
movies.to_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv', index=False)



In [ ]:
# Actors
unique_actors = set()
for sublist in movie['actors']:
    for actor in sublist:
        if actor.strip():  # Skip empty strings
            if '|' in actor:  # Only split if '|' exists
                try:
                    unique_actors.add(actor.split('|', 1)[1].strip())
                except IndexError:
                    print(f"Unexpected format in actor string: {actor}")
            else:
                unique_actors.add(actor.strip())  # If no '|', add the whole string

actors = pd.DataFrame({'name': list(unique_actors)})
actors.to_csv('/content/drive/MyDrive/Colab Notebooks/actors.csv', index=False)


In [ ]:
# Directors
unique_directors = set(director.strip() for sublist in movie['director'] for director in sublist if director.strip())
directors = pd.DataFrame({'name': list(unique_directors)})
directors.to_csv('/content/drive/MyDrive/Colab Notebooks/directors.csv', index=False)

In [ ]:
# Keywords
all_keywords = [kw for sublist in movie['keywords'] for kw in sublist]
unique_keywords = set(all_keywords)
keywords = pd.DataFrame({'name': list(unique_keywords)})
keywords.to_csv('/content/drive/MyDrive/Colab Notebooks/keywords.csv', index=False)

**Creating Seprate CSV for saving Relationship**

In [ ]:
# Movie-Actor Relationships
movie_actor = movie[['imdb-id', 'actors']].explode('actors').dropna()
movie_actor['actors'] = movie_actor['actors'].split('|', 1)[1].strip()
movie_actor.rename(columns={'imdb-id': 'imdb_id', 'actors': 'actor_name'}, inplace=True)
movie_actor.to_csv('/content/drive/MyDrive/Colab Notebooks/movie_actor.csv', index=False)

In [ ]:
# Movie-Director Relationships
movie_director = movie[['imdb-id', 'director']].explode('director').dropna()
movie_director['director'] = movie_director['director'].str.strip()
movie_director.rename(columns={'imdb-id': 'imdb_id', 'director': 'director_name'}, inplace=True)
movie_director.to_csv('/content/drive/MyDrive/Colab Notebooks/movie_director.csv', index=False)

In [ ]:
# Movie-Keyword Relationships
movie_keyword = movie[['imdb-id', 'keywords']].explode('keywords').dropna()
movie_keyword['keywords'] = movie_keyword['keywords'].str.strip()
movie_keyword.rename(columns={'imdb-id': 'imdb_id', 'keywords': 'keyword_name'}, inplace=True)
movie_keyword.to_csv('/content/drive/MyDrive/Colab Notebooks/movie_keyword.csv', index=False)

In [8]:
#Read files using pandas
movies = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movies.csv')
directors = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/directors.csv')
actors = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/actors.csv')
keywords = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/keywords.csv')
movie_actor = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_actor.csv')
movie_director = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_director.csv')
movie_keyword = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/movie_keyword.csv')


**Establishing Connection with Neo4j**

In [9]:
!pip install neo4j
from neo4j import GraphDatabase

driver = GraphDatabase.driver('neo4j+s://d9323d85.databases.neo4j.io', auth=('neo4j', 'pda7RyEuj8rwX0ZM1CazIMKHPaJ3aSdd2UbYCoCmPtM'), connection_timeout=120)
print('Connection established')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.3/312.3 kB 4.7 MB/s eta 0:00:00
Connection established


**Importing data from CSV to Neo4J nodes**

In [10]:
from tqdm import tqdm



In [ ]:
#---Create Director Node---
query = """
MERGE (d:Director {name: $name})
"""

for _, row in tqdm(directors.iterrows(), total=directors.shape[0]):
    with driver.session() as session: # Use a session within a context manager
        session.run(query, name=row['name'])

In [ ]:
# --- Create Movie Nodes ---
query_movies = """
MERGE (m:Movie {imdb_id: $imdb_id, title: $title, year: $year, runtime: $runtime, rating: $rating, votes: $votes, plot: $plot})
"""
for _, row in tqdm(movies.iterrows(), total=movies.shape[0], desc="Creating Movie Nodes"):
    with driver.session() as session:
        session.run(query_movies, imdb_id=row['imdb_id'], title=row['title'], year=row['year'], runtime=row['runtime'], rating=row['rating'], votes=row['votes'], plot=row['plot'])


In [ ]:
# --- Create Actor Nodes ---
query_actors = """
MERGE (a:Actor {name: $name})
"""
for _, row in tqdm(actors.iterrows(), total=actors.shape[0], desc="Creating Actor Nodes"):
    with driver.session() as session:
        session.run(query_actors, name=row['name'])



In [ ]:
# --- Create Keyword Nodes ---
query_keywords = """
MERGE (k:Keyword {name: $name})
"""
for _, row in tqdm(keywords.iterrows(), total=keywords.shape[0], desc="Creating Keyword Nodes"):
    with driver.session() as session:
        session.run(query_keywords, name=row['name'])



Creating Keyword Nodes: 100%|██████████| 29519/29519 [12:46<00:00, 38.52it/s]


**Importing Data from CSV to Neo4J reletionship**

In [ ]:
# --- Create Movie-Actor Relationships ---
query_movie_actor = """
MATCH (m:Movie {imdb_id: $imdb_id})
MATCH (a:Actor {name: $actor_name})
MERGE (m)-[:ACTED_BY]->(a)
MERGE (a)-[:ACTED_IN]->(m)
"""
for _, row in tqdm(movie_actor.iterrows(), total=movie_actor.shape[0], desc="Creating Movie-Actor Relationships"):
    with driver.session() as session:
        session.run(query_movie_actor, imdb_id=row['imdb_id'], actor_name=row['actor_name'])


Creating Movie-Actor Relationships:  96%|█████████▋| 35436/36783 [1:30:02<05:44,  3.91it/s]

In [ ]:
# --- Create Movie-Director Relationships ---
query_movie_director = """
MATCH (m:Movie {imdb_id: $imdb_id})
MATCH (d:Director {name: $director_name})
MERGE (m)-[:DIRECTED_BY]->(d)
MERGE (d)-[:DIRECTED_IN]->(m)
"""
for _, row in tqdm(movie_director.iterrows(), total=movie_director.shape[0], desc="Creating Movie-Director Relationships"):
    with driver.session() as session:
        session.run(query_movie_director, imdb_id=row['imdb_id'], director_name=row['director_name'])





In [ ]:
# --- Create Movie-Keyword Relationships ---
query_movie_keyword = """
MATCH (m:Movie {imdb_id: $imdb_id})
MATCH (k:Keyword {name: $keyword_name})
MERGE (m)-[:HAS_KEYWORD]->(k)
MERGE (k)-[:KEYWORD_IN]->(m)
"""
for _, row in tqdm(movie_keyword.iterrows(), total=movie_keyword.shape[0], desc="Creating Movie-Keyword Relationships"):
    with driver.session() as session:
        session.run(query_movie_keyword, imdb_id=row['imdb_id'], keyword_name=row['keyword_name'])


In [15]:
def run_query(query, parameters=None):  # Change default to None
    with driver.session() as session:
        result = session.run(query, parameters=parameters)  # Pass parameters here
        return pd.DataFrame([dict(record) for record in result])

# Node Counts
node_counts = run_query("""
MATCH (n)
RETURN labels(n) AS Labels, COUNT(*) AS Count
""")

# Relationship Counts
relationship_counts = run_query("""
MATCH ()-[r]->()
RETURN type(r) AS RelationshipType, COUNT(*) AS Count
""")

In [16]:
def get_actor_based_recommendations(movie_title, top_n=3):
    query = """
    MATCH (m:Movie {title: $movie_title})<-[:ACTED_IN]-(a:Actor)-[:ACTED_IN]->(rec:Movie)
    WHERE m <> rec
    WITH rec, COUNT(a) AS shared_actors
    RETURN rec.title AS RecommendedMovie,
           rec.rating AS Rating,
           rec.year AS Year,
           shared_actors
    ORDER BY shared_actors DESC, rec.rating DESC
    LIMIT $top_n
    """
    parameters = {'movie_title': movie_title, 'top_n': top_n}
    df = run_query(query, parameters)
    return df

# Example usage
actor_recommendations = get_actor_based_recommendations("PK")
print(actor_recommendations)

       RecommendedMovie  Rating    Year  shared_actors
0   Munna Bhai M.B.B.S.     8.1  (2003)              2
1  Lage Raho Munna Bhai     8.0  (2006)              2
2              3 Idiots     8.4  (2009)              1
